# 🏆 Run #5 — QLoRA-DoRA SFT: Inyección de Gramática Netelpro en Qwen2.5-1.5B-Instruct

**Qwen/Qwen2.5-1.5B-Instruct + QLoRA-DoRA (Weight-Decomposed Low-Rank Adaptation)**  
Dataset: `netelpro_grammar_v3` (`train_mix.jsonl` + `heldout.jsonl`) verificado por el compilador real de Netelpro.

---

### Diagnóstico de la falla del Run #4 y Correcciones del Run #5
| Componente | Run #4 (Goliath-Killer v2 - Falló en inyectar) | Run #5 (Este Notebook - QLoRA-DoRA) |
|---|---|---|
| **Arquitectura** | OLMoE-1B-7B (MoE disperso, 64 expertos) | **Qwen2.5-1.5B-Instruct** (denso, disciplinado en 1.5B) |
| **Adaptador** | LoRA estándar (r=16, alpha=32) | **DoRA** (`use_dora=True`, r=128, alpha=256, dropout=0.05) |
| **Módulos objetivo** | Solo atención (`q, k, v, o`) | **Atención + MLP completo** (`q, k, v, o, gate, up, down`) |
| **Capacidad de inyección** | Inyección insuficiente; los pesos base diluyeron la sintaxis | Descomposición de magnitud y dirección (DoRA) a alto rango |
| **Dataset** | Contratos de 12 familias (mezcla dispersa) | `netelpro_grammar_v3` curado (`train_mix` + `heldout`) |
| **Gate Post-Entrenamiento** | Solo benchmark pasivo | **GATE CELDA BLOQUEANTE** (real Netelpro compiler, fail-closed) |

---

### Configuración de la sesión en Kaggle (panel lateral derecho)
- **Accelerator:** GPU T4 x2 (usaremos GPU 0 para entrenamiento eficiente en single-GPU)
- **Internet:** **On** (imprescindible para clonar `netelpro` e instalar dependencias)
- **Tiempo estimado:** ~45–60 min para 2 épocas en T4 (batch efectivo = 16, seq = 1024)

## 1. Instalación de dependencias (Pins T4-compatibles)

> [!IMPORTANT]
> Kaggle corre sobre Python 3.12 y controladores CUDA recientes. Siguiendo la calibración de `train_goliath_killer_v2_kaggle.ipynb`, desinstalamos `bitsandbytes` e instalamos la versión `0.50.2` que no rompe el kernel NF4. Usamos exclusivamente `%pip` (**nunca** `!pip`).

In [ ]:
# Kaggle usa Python 3.12 y CUDA reciente. bitsandbytes 0.45.2 NO es compatible.
# Usamos %pip con versiones exactamente fijadas y compatibles con GPU T4.
%pip uninstall -y bitsandbytes
%pip install -q --no-cache-dir "bitsandbytes==0.50.2"
%pip install -q --upgrade "transformers==4.48.3" "trl==0.15.2" "peft==0.14.0" "accelerate==1.3.0" "datasets==3.2.0" "llvmlite>=0.49" sentencepiece huggingface_hub tabulate gguf

print("✅ Dependencias instaladas exitosamente con %pip.")

## 2. Diagnóstico de hardware y clonación del repositorio Netelpro

Verificamos la GPU T4, probamos el kernel NF4 de `bitsandbytes` con un tensor de prueba, y clonamos `github.com/jona2428/netelpro` para tener el motor de compilación nativo y los módulos de evaluación disponibles en `sys.path`.

In [ ]:
import os
import sys
import gc
import subprocess
from pathlib import Path

# Configuración de memoria y logging de CUDA
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import bitsandbytes as bnb

print("=" * 65)
print("DIAGNÓSTICO DE HARDWARE & ENTORNO KAGGLE")
print("=" * 65)
print(f"PyTorch: {torch.__version__} | CUDA disponible: {torch.cuda.is_available()}")
print(f"bitsandbytes: {bnb.__version__} | CUDA de PyTorch: {torch.version.cuda}")
num_gpus = torch.cuda.device_count()
print(f"GPUs detectadas: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    free = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(f"  GPU {i}: {name} | Total: {vram:.2f} GB | Libre: {free:.2f} GB")

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU activa. En Kaggle: Settings > Accelerator > GPU T4 x2.")

# Probar el kernel NF4 de bitsandbytes antes de descargar el modelo
try:
    probe = torch.randn(256, device="cuda", dtype=torch.float16)
    packed, state = bnb.functional.quantize_4bit(probe, quant_type="nf4")
    del probe, packed, state
    torch.cuda.empty_cache()
    print("Kernel NF4 de bitsandbytes: OK")
except Exception as exc:
    raise RuntimeError(
        "bitsandbytes no cargó el kernel CUDA/NF4. "
        f"Error original: {type(exc).__name__}: {exc}"
    ) from exc

# Clonar repo github.com/jona2428/netelpro
REPO_DIR = Path("/kaggle/working/netelpro")
if not REPO_DIR.exists():
    print(f"Clonando https://github.com/jona2428/netelpro.git en {REPO_DIR}...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/jona2428/netelpro.git", str(REPO_DIR)],
        check=True
    )
else:
    print(f"Repo {REPO_DIR} ya existe. Actualizando con git pull...")
    try:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print(f"Nota (git pull): {e}")

# Agregar repo y subcarpetas al sys.path
for path_cand in [str(REPO_DIR), str(REPO_DIR / "src")]:
    if path_cand not in sys.path:
        sys.path.insert(0, path_cand)

# Verificar que el compilador real de Netelpro responda
from netelpro.rule_filter import RuleFilterError, compile_filter
from benchmarks.run5_gate import extract_block, compile_verdict, gate_report

probe_verdict = compile_verdict("""```netelpro
(truth-table filter-rule
  (a : (Int 0 1))
  ((0) -> 0)
  ((1) -> 1)
  ((_) -> 0))
```""")
assert probe_verdict.ok, f"Fallo en sonda de compilador: {probe_verdict.error}"
print("✅ Motor Netelpro real (LLVM) y módulo benchmarks.run5_gate cargados y operativos.")

## 3. Carga de datos de entrenamiento y held-out (Netelpro Grammar v3)

Cargamos `training/data/netelpro_grammar_v3/train_mix.jsonl` y `heldout.jsonl` desde el repositorio clonado.
Si cualquiera de los archivos no está presente o está vacío, el notebook falla inmediatamente de forma visible.

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

DATA_DIR = REPO_DIR / "training" / "data" / "netelpro_grammar_v3"
TRAIN_FILE = DATA_DIR / "train_mix.jsonl"
HELDOUT_FILE = DATA_DIR / "heldout.jsonl"

print("Comprobando existencia de los datasets de gramática v3:")
print(f"  Train:   {TRAIN_FILE}")
print(f"  Heldout: {HELDOUT_FILE}")

if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"DATASET_MISSING: El archivo de entrenamiento '{TRAIN_FILE}' no existe en el repositorio clonado. "
        "Asegúrate de que 'training/data/netelpro_grammar_v3/train_mix.jsonl' haya sido commiteado en el repo."
    )
if not HELDOUT_FILE.exists():
    raise FileNotFoundError(
        f"DATASET_MISSING: El archivo held-out '{HELDOUT_FILE}' no existe en el repositorio clonado. "
        "Asegúrate de que 'training/data/netelpro_grammar_v3/heldout.jsonl' haya sido commiteado en el repo."
    )

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f if line.strip()]

with open(HELDOUT_FILE, "r", encoding="utf-8") as f:
    heldout_data = [json.loads(line) for line in f if line.strip()]

if not train_data:
    raise ValueError(f"DATASET_EMPTY: '{TRAIN_FILE}' está vacío.")
if not heldout_data:
    raise ValueError(f"DATASET_EMPTY: '{HELDOUT_FILE}' está vacío.")

print(f"✅ Datasets cargados: {len(train_data)} train | {len(heldout_data)} heldout")
sample_prompt = train_data[0].get("prompt") or train_data[0].get("instruction") or ""
print(f"Muestra prompt train [0]: {sample_prompt[:120]}...")

## 4. Carga de Qwen2.5-1.5B en 4-bit NF4 y Configuración DoRA

- **Modelo base:** `Qwen/Qwen2.5-1.5B-Instruct`
- **Cuantización:** 4-bit NF4 con doble cuantización (`bnb_4bit_use_double_quant=True`), `compute_dtype=torch.float16` (la GPU T4 no cuenta con soporte nativo para bf16).
- **PEFT DoRA:** `use_dora=True`, `r=128`, `lora_alpha=256` (ratio 2:1), `lora_dropout=0.05`.
- **Target modules:** los 7 módulos de proyección (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
- **Chat Template:** Aplicamos exactamente el template de evaluación usando el `HONESTY_SYSTEM_PROMPT` oficial importado de `benchmarks/vtb_ood_runner.py`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from benchmarks.vtb_ood_runner import HONESTY_SYSTEM_PROMPT

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Cargando tokenizer para {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit NF4 + double quant, compute dtype fp16 para GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Cargando {MODEL_ID} en 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# Configuración DoRA de alto rango: r=128, alpha=256, dropout=0.05
# Se abarcan todas las proyecciones de atención y MLP
peft_config = LoraConfig(
    use_dora=True,
    r=128,
    lora_alpha=256,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)
model = get_peft_model(model, peft_config)

print("\nParámetros entrenables con DoRA:")
model.print_trainable_parameters()

# Formateo con chat template idéntico al de evaluación oficial (vtb_ood_runner)
def format_chat_example(example):
    prompt_text = example.get("prompt") or example.get("instruction") or example.get("input") or ""
    completion_text = example.get("completion") or example.get("output") or example.get("response") or ""
    messages = [
        {"role": "system", "content": HONESTY_SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]
    chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return chat_prompt + completion_text + (tokenizer.eos_token or "")

train_texts = [format_chat_example(ex) for ex in train_data]
formatted_train_dataset = Dataset.from_dict({"text": train_texts})
print(f"Dataset de entrenamiento formateado: {len(formatted_train_dataset)} ejemplos.")

## 5. Entrenamiento SFTTrainer con DoRA y Manejo de Checkpoints

### Especificaciones de Entrenamiento
- **Épocas:** `EPOCHS = 2` (constante configurable al inicio de la celda).
- **Batch Size:** `per_device_train_batch_size = 2` con `gradient_accumulation_steps = 8` ⇒ batch efectivo = 16.
- **Learning Rate:** `2e-4` con `lr_scheduler_type="cosine"` y `warmup_ratio=0.03`.
- **Secuencia:** `max_seq_length = 1024`.
- **Precisión:** `fp16 = True`, `bf16 = False`.
- **Optimizador:** `paged_adamw_8bit`.
- **Gradient Checkpointing:** activado con `use_reentrant=False`.
- **Semilla:** fija (`seed = 42`).

### Persistencia y Reanudación (Checkpoints)
Los checkpoints se guardan al final de cada época en `/kaggle/working/qwen25_run5_checkpoints`.
Si la sesión de Kaggle se interrumpe, se puede reanudar pasando `resume_from_checkpoint=True` a `trainer.train()`. Al terminar las 2 épocas, el adaptador final consolidado se guarda en `/kaggle/working/qwen25_run5_adapter`.

In [ ]:
from trl import SFTConfig, SFTTrainer

# Constante de épocas configurable
EPOCHS = 2
OUTPUT_DIR = "/kaggle/working/qwen25_run5_checkpoints"
ADAPTER_DIR = "/kaggle/working/qwen25_run5_adapter"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    bf16=False,
    max_seq_length=1024,
    dataset_text_field="text",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train_dataset,
    processing_class=tokenizer,
)

print("=" * 65)
print("INICIANDO ENTRENAMIENTO QLORA-DORA (RUN #5)")
print(f"Modelo: {MODEL_ID} | DoRA r=128 alpha=256 | Épocas: {EPOCHS}")
print(f"Batch efectivo: {2 * 8} | Max seq length: 1024 | LR: 2e-4 cosine")
print("=" * 65)

# Para reanudar una sesión previa interrumpida: trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
print(f"Entrenamiento completado en {train_result.metrics.get('train_runtime', 0) / 60:.1f} minutos.")

# Guardar adaptador consolidado y tokenizer
model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adaptador DoRA guardado exitosamente en: {ADAPTER_DIR}")

## 6. Celda GATE (Mandatoria Post-Entrenamiento)

> [!CAUTION]
> **COMPILACIÓN ESTRICTA FAIL-CLOSED**: El modelo no puede avanzar a la exportación GGUF ni a la publicación en Hugging Face si no demuestra haber inyectado la gramática formal.
> 
> Esta celda ejecuta $\ge 8$ generaciones sobre el split `heldout_data` (semilla determinista por muestra, `temp = 0.8`, `max_new_tokens = 600`), extrae el bloque de contrato y casos de verificación con la misma extracción de `benchmarks/principle_bench.py` y `run5_gate.py`, y compila cada uno con el **compilador nativo real de Netelpro** (`compile_filter`).
> 
> Si la tasa de compilación es **0%**, se levanta una excepción visible `GATE_FAILED` que detiene la ejecución inmediatamente.

In [ ]:
# ============================================================
# GATE CELL (MANDATORIA POST-ENTRENAMIENTO)
# ============================================================
import torch
from benchmarks.run5_gate import extract_block, compile_verdict, gate_report

model.eval()

# Función de generación con el protocolo oficial del benchmark
def generate_heldout_sample(prompt: str, seed: int, max_new_tokens: int = 600, temp: float = 0.8) -> str:
    messages = [
        {"role": "system", "content": HONESTY_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    torch.manual_seed(seed)
    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temp,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

NUM_GATE_SAMPLES = max(8, min(16, len(heldout_data)))
print(f"Ejecutando GATE sobre {NUM_GATE_SAMPLES} ejemplos held-out (temp=0.8, seed por muestra, max_new_tokens=600)...")

gate_verdicts = []
for idx in range(NUM_GATE_SAMPLES):
    item = heldout_data[idx]
    prompt = item.get("prompt") or item.get("instruction") or ""
    seed = 42 + idx
    
    # Generación con temperatura 0.8 y semilla fijada por muestra
    response = generate_heldout_sample(prompt, seed=seed, max_new_tokens=600, temp=0.8)
    
    # Extracción y compilación con el compilador real de Netelpro (run5_gate)
    verdict = compile_verdict(response)
    gate_verdicts.append(verdict)
    
    status_icon = "PASS ✅" if verdict.ok else "FAIL ❌"
    detail = "Compilación exitosa" if verdict.ok else (verdict.error or "Error desconocido")
    print(f"[{idx+1:02d}/{NUM_GATE_SAMPLES:02d}] Seed {seed}: {status_icon} | {detail[:70]}")

# Reporte consolidado con tabla y tasa de compilación
report = gate_report(gate_verdicts, raise_on_zero=False)
print("\n" + report["report_text"])

# Verificación de compuerta: Si ningún contrato compila (rate == 0), levantar GATE_FAILED
if report["compile_rate"] == 0.0:
    raise RuntimeError(
        "GATE_FAILED: La tasa de compilación en el held-out es 0.0%. "
        "El modelo no logró inyectar la gramática formal Netelpro. "
        "Abortando exportación y publicación."
    )

print(f"🎉 GATE APROBADO: Tasa de compilación {report['compile_rate_pct']:.1f}% > 0.0%. Continuando a exportación GGUF...")

## 7. Exportación del Modelo Fusionado (fp16), GGUF Q4_K_M y Publicación en Hugging Face

1. **Fusión en fp16:** Se fusiona el adaptador DoRA en el modelo base limpio en precisión fp16 y se guarda en `/kaggle/working/qwen25_15b_run5_dora_merged`.
2. **Exportación GGUF Q4_K_M:** Se exporta a formato GGUF siguiendo el método directo de los notebooks RAFT previos (`q4_k_m`).
3. **Subida a Hugging Face Hub:** Si se configuró `HF_TOKEN` en **Kaggle Secrets** (`Add-ons > Secrets`), se crea el repositorio y se suben el modelo fusionado y el archivo GGUF automáticamente.

In [ ]:
# ============================================================
# EXPORT CELL: MERGE FP16 + GGUF Q4_K_M + HF UPLOAD
# ============================================================
import os
import gc
import shutil
import subprocess
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_DIR = "/kaggle/working/qwen25_15b_run5_dora_merged"
GGUF_DIR = "/kaggle/working/qwen25_15b_run5_gguf"
GGUF_FILE = Path(GGUF_DIR) / "qwen2.5-1.5b-run5-netelpro-q4_k_m.gguf"

# 1. Liberar memoria del modelo de entrenamiento
del trainer, model
torch.cuda.empty_cache()
gc.collect()

print("1. Fusionando adaptador DoRA con el modelo base en fp16...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

del base_model, merged_model
torch.cuda.empty_cache()
gc.collect()
print(f"✅ Modelo fusionado (fp16) guardado en: {MERGED_DIR}")

# 2. Conversión a GGUF Q4_K_M
print("\n2. Exportando a GGUF Q4_K_M...")
os.makedirs(GGUF_DIR, exist_ok=True)

LLAMA_CPP_DIR = Path("/kaggle/working/llama.cpp")
if not LLAMA_CPP_DIR.exists():
    print("Clonando llama.cpp para conversión de pesos...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)], check=True)

# Requisitos para el script de conversión
%pip install -q -r /kaggle/working/llama.cpp/requirements.txt

# Convertir de formato HF a GGUF fp16 intermedio
fp16_gguf = Path(GGUF_DIR) / "model-fp16.gguf"
print("Ejecutando convert_hf_to_gguf.py...")
subprocess.run([
    sys.executable,
    str(LLAMA_CPP_DIR / "convert_hf_to_gguf.py"),
    MERGED_DIR,
    "--outfile", str(fp16_gguf),
    "--outtype", "f16",
], check=True)

# Compilar herramienta llama-quantize
print("Compilando llama-quantize...")
build_dir = LLAMA_CPP_DIR / "build"
subprocess.run(["cmake", "-B", str(build_dir), str(LLAMA_CPP_DIR)], check=True)
subprocess.run(["cmake", "--build", str(build_dir), "--target", "llama-quantize", "-j4"], check=True)
quantize_bin = build_dir / "bin" / "llama-quantize"

# Cuantizar a Q4_K_M
print(f"Cuantizando a Q4_K_M -> {GGUF_FILE}...")
subprocess.run([str(quantize_bin), str(fp16_gguf), str(GGUF_FILE), "q4_k_m"], check=True)

# Remover fp16 intermedio para optimizar disco en Kaggle
if fp16_gguf.exists():
    fp16_gguf.unlink()

gguf_size_mb = GGUF_FILE.stat().st_size / (1024 * 1024)
print(f"✅ Exportación GGUF completada: {GGUF_FILE} ({gguf_size_mb:.1f} MB)")

# 3. Publicación opcional en Hugging Face
print("\n3. Publicación en Hugging Face Hub (usando Kaggle Secrets)...")
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    HF_REPO = "JonaECG/Qwen2.5-1.5B-Netelpro-Grammar-Run5"
    print(f"Publicando en repo HF: {HF_REPO}...")
    api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=MERGED_DIR, path_in_repo="merged_fp16", repo_id=HF_REPO)
    api.upload_file(path_or_fileobj=str(GGUF_FILE), path_in_repo=GGUF_FILE.name, repo_id=HF_REPO)
    print(f"🎉 Modelo publicado con éxito: https://huggingface.co/{HF_REPO}")
else:
    print("ℹ️ HF_TOKEN no detectado en Kaggle Secrets. Subida omitida.")
    print(f"Archivos disponibles en /kaggle/working/: {GGUF_FILE.name}")

## 8. Enlaces de descarga local en Kaggle

In [ ]:
from IPython.display import FileLink, display

for file_path in [GGUF_FILE, Path(ADAPTER_DIR) / "adapter_model.safetensors"]:
    if file_path.exists():
        display(FileLink(str(file_path)))